# Choosing a Model: A Custom Model Eval Case Study
*A complete worked example of selecting the best model for a real Text-to-SQL application — gathering requirements and estimating cost (including prompt caching and blended pricing), shortlisting candidates from a coding leaderboard, and running a full custom evaluation across five models to reach a final decision*

## From theory to a real worked example

Everything covered so far about model evals — benchmarks, custom evals, leaderboards, why lab-published numbers deserve skepticism — sets up one practical skill: actually choosing the right model for a real application. This notebook walks through a complete, concrete example of that process end to end, using a genuine product scenario rather than an abstract description of the steps.

## The case study: a natural-language query feature for a cricket website

Consider a large cricket information and live-scores website — the kind millions of fans check during a match for ball-by-ball updates and commentary. During big matches, the site's commentary team fields a constant stream of viewer questions — "What's Virat Kohli's batting average against Pakistan?" or "How many wickets did Jasprit Bumrah take last time these two teams played?" — and a team of human analysts sits alongside the commentary team, manually writing database queries to answer each one on the spot, which the commentary team then feeds back into the live text feed.

This works, but it doesn't scale. During a high-profile match, the volume of incoming questions can spike to hundreds within minutes — far more than a small analyst team can keep up with by hand. The proposed fix: build a feature that lets any fan type a cricket question directly into the site, and have the system generate an answer automatically, with no analyst in the loop at all.

## What this system actually is: Text-to-SQL, not RAG

The feature works like this: a user submits a plain-English question. That question, along with the database's schema, gets sent to an LLM configured specifically to generate SQL. The LLM returns a single SQL query, which the system then runs directly against the database, and whatever the query returns is shown back to the user.

This is a **Text-to-SQL** application, not a RAG system — an important distinction. A RAG system retrieves relevant external documents at query time and hands them to the model as context. Here, nothing is retrieved — the model is simply given the database's schema (which stays fixed every time) directly in its prompt, and it's up to the model to translate the natural-language question into a correct SQL query against that fixed structure. No retrieval step exists at all.

## The three-stage model-selection process

Choosing the LLM that will power this feature — effectively the "brain" of the whole system — follows the same three-stage process covered when leaderboards were introduced:

1. **Gather requirements.** Write down exactly what the application needs before looking at a single model.
2. **Shortlist candidates using leaderboards.** Use the requirements to narrow the field down to 5–10 serious candidate models — remembering that leaderboards are a filtering tool, not a decision tool.
3. **Run a custom model eval on the shortlist.** Test those candidates directly against real data from this specific application, and pick whichever one performs best.

The rest of this notebook works through Stage 1 in full detail, using this Text-to-SQL feature as the concrete example.

## Stage 1: gathering requirements

**The task, stated precisely.** The model needs to generate a single, correct SQL query from a natural-language cricket question and a fixed database schema — nothing more elaborate than that.

**Why the business wants this feature at all.** Two distinct reasons sit behind it. The first is scalability: a manual analyst team simply can't keep pace with question volume during a major match, and an automated system removes that bottleneck entirely. The second is less obvious but arguably more important: **retention**. Almost every content website's core incentive is keeping users on the site longer, since more time on-site generally means more ad impressions and more revenue. Letting a fan type a question and get an instant answer is a small but effective way to keep them engaged with the page rather than leaving to search elsewhere — the feature isn't purely a cost-saving measure, it's also a deliberate engagement mechanism.

**The cost ceiling.** Before any model gets selected, the business sets a budget the feature has to operate within — in this case, a monthly ceiling of roughly ₹3 lakh. Whatever model gets chosen has to keep the feature's running cost under that number.

## Estimating cost: the prompt and the schema

Before any model can be evaluated for capability, its cost needs to be estimated — and for an LLM-based system, the dominant cost is almost always the per-token price charged by the model provider.

Every request sent to the model follows a fixed system prompt along these lines:

> You are a Text-to-SQL generator. Given a database schema and a question, return a single SQL query that answers it, using SQLite syntax. Return only the SQL query — no explanation.

The schema itself — scoped to one specific tournament to keep things manageable, in this case the IPL — consists of two tables: a **matches** table (which teams played, the ground, the date, the umpires) and a **deliveries** table (ball-by-ball data: batter, bowler, runs scored, wickets, and so on for every single delivery of every match). This schema is included in every request and never changes.

A representative question sent by a user:

> Which bowler has the best economy rate among bowlers who have bowled at least 500 legal balls?

The system prompt, the schema, and the question together form the full input sent to the model; the model's entire output is just the generated SQL query.

## Estimating cost: counting tokens and running the numbers

Actually counting the tokens in the combined system prompt, schema, and a typical question comes out to roughly **400 input tokens** per request. The generated SQL query in response comes out to roughly **100 output tokens**. Since the system prompt and schema are fixed and only the user's question changes, this token count stays fairly stable across different queries.

The other number needed is **query volume** — how many questions the feature will realistically need to answer per day. Traffic here is genuinely spiky — it clusters heavily around major matches and stays much quieter otherwise — but for a working estimate, a reasonable planning assumption is around **50,000 questions per day**.

With those two numbers in hand, monthly cost for any candidate model can be estimated directly:

$$\text{Cost per query} = \left(\frac{\text{input tokens}}{1{,}000{,}000} \times \text{input price}\right) + \left(\frac{\text{output tokens}}{1{,}000{,}000} \times \text{output price}\right)$$

$$\text{Monthly cost} = \text{Cost per query} \times \text{queries per day} \times 30$$

## Worked example: a premium model turns out too expensive

Suppose the first candidate under consideration — call it **Model A** — is a strong, premium-tier model priced at $10 per million input tokens and $50 per million output tokens.

**Cost per query:**

$$\left(\frac{400}{1{,}000{,}000} \times 10\right) + \left(\frac{100}{1{,}000{,}000} \times 50\right) = 0.004 + 0.005 = \$0.009$$

**Daily cost** (50,000 queries/day):

$$0.009 \times 50{,}000 = \$450 \text{ per day}$$

**Monthly cost** (30 days):

$$450 \times 30 = \$13{,}500 \text{ per month}$$

Converting to rupees at roughly ₹95 per dollar:

$$13{,}500 \times 95 \approx ₹12.8 \text{ lakh per month}$$

Against a budget ceiling of ₹3 lakh per month, Model A comes in at more than **four times** the available budget. However capable Model A might be, it's simply not affordable for this feature at this query volume — which means the search has to move to meaningfully cheaper alternatives, roughly a quarter of Model A's price, before capability can even be evaluated.

## A cost optimization worth knowing: prompt caching

Before moving past cost, one optimization is worth understanding, since it can meaningfully change the numbers above. Look closely at the request structure for this application: the system prompt and the database schema are **identical across all 50,000 queries** answered in a day — the only part that actually changes from one request to the next is the user's specific question. Paying full input-token price for that same repeated block, tens of thousands of times a day, is wasteful.

This is exactly what **prompt caching** addresses. Providers let a portion of a prompt be cached on their servers, so that identical or repeated content doesn't need to be paid for at full price on every subsequent request — only the first time it's sent.

### How cached pricing typically works

Providers commonly offer a couple of caching duration options, each with its own pricing:

- **A short cache window (e.g., 5 minutes).** Writing content into this cache costs *more* than a normal input token would — since the provider is also doing the work of storing it, not just processing it. But any subsequent request within that window that reuses the same cached content pays a much lower rate for that portion — often around a tenth of the normal input price. If no new request reuses the cache within the window, it expires, and the next request has to pay the full write cost again to re-cache it.
- **A longer cache window (e.g., 1 hour).** Writing to this cache costs more upfront than the short window does, but the content stays available for reuse over a much longer stretch — useful when requests come in less frequently than every few minutes.

Choosing between them comes down to how often new requests actually arrive. For an application handling 50,000 queries a day, new questions arrive constantly, so a short cache window works well — the cache rarely has time to expire before the next question reuses it, aside from genuinely quiet stretches like the middle of the night. For a much lower-traffic application — say, only 1,000 questions a day — requests might be spaced out enough that a short cache window expires between uses far more often, making the longer window the more cost-effective choice instead.

### Estimated impact on cost

Applying prompt caching to the earlier Model A cost estimate — where the system prompt and schema make up the bulk of the roughly 400 input tokens per query, and that content repeats identically across nearly every request — could plausibly cut the estimated monthly cost roughly in half, since the large majority of input tokens on most queries would be served from cache at a steep discount rather than priced as fresh input every time.

This optimization is highly scenario-dependent, though. It works well here specifically because the system prompt and schema are large, fixed, and repeated verbatim across requests. In a RAG chatbot, by contrast, a large portion of the prompt is the *retrieved context* — and that context is different on essentially every query, since it depends on what gets retrieved for that specific question. There's much less repeated content to actually cache in that setting, so prompt caching delivers far less benefit there than it does for an application like this one, where the bulk of the prompt is genuinely static.

### A brief technical note

Under the hood, this optimization relies on a concept called **KV caching** (key-value caching), tied to the attention mechanism inside transformer models — the same K and V vectors computed during attention can be reused for repeated content rather than recomputed from scratch every time. The mechanics of attention and KV caching are a topic in their own right, worth returning to separately; the practical takeaway here is simply that this is *why* prompt caching is possible at the infrastructure level.

### A practical note on schema descriptions

If a database's column names aren't self-explanatory — highly domain-specific or abbreviated naming, for instance — nothing stops adding a further paragraph to the system prompt describing what each column actually represents, beyond just listing the schema structure. This makes the system prompt longer, which costs somewhat more per query — but since that description would also be part of the same large, repeated, cacheable block covered above, prompt caching offsets a meaningful portion of that added cost.

## Requirement: latency

Whether latency matters at all depends entirely on the application. A research assistant that goes off and gathers information from the web has little reason to respond instantly — a user asking a research assistant a complex question generally expects to wait. This application is the opposite case: someone typing a question mid-match, driven by in-the-moment curiosity during a live game, expects a fast answer.

A reasonable target here is **2 to 3 seconds**. Beyond that, users start to feel like something's gone wrong. This requirement rules out certain categories of models outright — anything with notably high latency, or a reasoning model that spends significant time thinking before producing an answer, isn't a fit for this specific application, regardless of how capable it otherwise is.

## Requirement: context window

Context window size turns out not to matter much here. Every question a user asks is a single, standalone request — there's no ongoing conversation, no multi-turn back-and-forth, and no dependency on anything asked earlier. Each query is fully disjoint from the last.

This is a meaningful contrast with other kinds of applications: a chatbot or a coding assistant, where conversation history or a large codebase needs to stay in context across many turns, genuinely depends on a large context window. Here, since there's no session state to carry forward, even a model with a comparatively small context window works fine — this requirement simply isn't a filtering criterion for this particular application.

## Requirement: deployment constraints

The next question is whether this application needs to run on self-hosted, on-premise infrastructure — often a requirement when handling sensitive or private data that can't leave a company's own servers — or whether a public API is acceptable.

Here, there's no such constraint: nothing about cricket statistics and match records counts as sensitive data that needs to stay on private infrastructure. That means a public API from a provider like Anthropic or OpenAI is not just acceptable but arguably preferable — public APIs are generally more reliable and better engineered for uptime than a self-managed deployment would be, which would also require a dedicated technical team to build and maintain.

## Requirement: correctness

Correctness matters enormously for this application, for a very specific reason: cricket fans are notoriously precise about statistics and records. If a query about who scored the most runs in a tournament returns the wrong player's name, that mistake doesn't stay quiet — a screenshot of the error is exactly the kind of thing that gets shared widely on social media, embarrassing the site publicly and damaging its credibility. Hallucinating or generating an incorrect answer isn't a tolerable failure mode here the way it might be in a lower-stakes application.

Since SQL generation is fundamentally a coding task, this correctness requirement can be checked two ways: through a coding-focused leaderboard as an initial signal, and — more importantly — through a custom eval built directly around this application's own schema and query patterns, which is the only way to know how a candidate model actually performs on *this* specific task.

## Requirements, summarized

Putting the full picture together, here's what Stage 1 established for this application:

- **Task:** Text-to-SQL generation.
- **Cost ceiling:** roughly ₹3 lakh per month.
- **Latency:** should not exceed 2–3 seconds.
- **Context window:** not a meaningful factor — no multi-turn conversation exists.
- **Deployment:** no on-premise constraint; a public API is preferred.
- **Correctness:** a high priority, given how closely cricket fans scrutinize factual accuracy.

Having these requirements written down and settled *before* opening a single leaderboard is exactly what keeps the next step objective — there's no temptation to simply pick whichever model sits at the top of a general leaderboard, because the actual constraints of this specific application are already clear.

## Stage 2: shortlisting candidates from a leaderboard

With requirements settled, the next question is which leaderboard to actually consult. The ideal case would be a dedicated Text-to-SQL leaderboard — a benchmark built specifically for exactly this task.

### Why dedicated Text-to-SQL leaderboards didn't work here

A couple of well-known Text-to-SQL benchmarks do exist — BIRD-SQL and Spider are the best-known examples — and checking them directly seemed like the obvious first move. In practice, neither turned out usable for this purpose, for two consistent reasons:

- **They weren't current.** The models listed were noticeably behind the current generation, with no recent frontier models represented at all.
- **Unclear or mismatched methodology.** Some of the ranked entries appeared to be models specifically fine-tuned for SQL generation, or entire evaluation harnesses rather than general-purpose LLMs — not the kind of general frontier model this application actually needs to choose between. One leaderboard's underlying evaluation dataset also wasn't clearly documented, making it hard to trust the numbers at all.

This is a direct, practical instance of the "read the leaderboard correctly" guidance covered earlier — a leaderboard that isn't updated, or whose methodology isn't transparent, isn't one worth trusting blindly, no matter how directly relevant its name sounds.

### The fallback: using a coding leaderboard as an alias

Since no trustworthy, up-to-date Text-to-SQL leaderboard was available, the fallback was to use a **coding leaderboard** instead — reasoning that generating SQL is fundamentally a coding activity, so general coding capability serves as a reasonable proxy (an "alias") for SQL-generation capability specifically.

The specific leaderboard used here was a third-party aggregator site that pulls data from multiple underlying coding benchmarks, combines them into one overall rating per model, and also reports cost, throughput (speed), and context window alongside that rating — the kind of multi-benchmark, general-capability leaderboard covered earlier as the most broadly useful type. It listed 146 models in total, and — critically — it was current, with the most recent frontier models from every major provider represented.

### Reading the fine print: blended pricing

This particular leaderboard reports a single combined price per model, rather than separate input and output rates — and understanding exactly how that single number was derived matters before using it for anything. The leaderboard's own documentation specifies a **blended price**, computed using a fixed input-to-output ratio — in this case, 4:1.

Applying that to Model A's actual rates (recall: $10 per million input tokens, $50 per million output tokens):

$$\text{Blended price} = \frac{(4 \times 10) + (1 \times 50)}{4 + 1} = \frac{40 + 50}{5} = \frac{90}{5} = \$18 \text{ per million tokens}$$

That $18 figure is exactly what the leaderboard displays for Model A. Reading that fine print — figuring out what ratio a "blended" number actually assumes — is precisely the kind of detail the earlier guidance on reading leaderboards correctly was pointing at.

### A convenient coincidence: the ratio matches exactly

This particular Text-to-SQL application sends roughly 400 input tokens and receives roughly 100 output tokens per query — a 4:1 ratio, identical to the leaderboard's own blending assumption. That means the blended price can be used directly, with no further adjustment needed, to estimate this specific application's cost for any model on the leaderboard:

$$\text{Monthly cost} = \frac{\text{total tokens per query}}{1{,}000{,}000} \times \text{blended price} \times \text{queries per day} \times 30$$

With 500 total tokens per query (400 input + 100 output) and 50,000 queries a day, this single formula, applied across all 146 models' blended prices, produces a monthly cost estimate for every candidate at once — without needing to separately track input and output pricing for each one.

### Filtering and ranking the full leaderboard

With that formula confirmed against Model A's own numbers, the same calculation was applied across the entire leaderboard — not just the one illustrative example, but the full pool of 146 candidate models — giving a monthly cost estimate for every single one of them at once. The first filtering step was straightforward: reject every model whose estimated monthly cost exceeded the budget ceiling, leaving a much smaller pool of affordable candidates.

The remaining candidates then needed to be ranked using two factors: coding capability (the leaderboard's overall rating) and speed (characters printed per second). Both were **normalized to a 0–1 scale** using min-max normalization — subtracting the minimum value in each column and dividing by the range — so the two, originally on very different scales, could be combined fairly. The combined score used a **90/10 weighting**:

$$\text{Score} = 0.9 \times \text{normalized rating} + 0.1 \times \text{normalized speed}$$

Capability was weighted far more heavily than speed for a specific reason: this application's output is just a short generated SQL query — around 100 tokens. Printing 100 tokens takes a trivial amount of time even for a comparatively slow model; speed would matter far more if the expected output were long-form, like a multi-paragraph essay. Since output length here is small regardless of which model is used, raw generation speed has little practical effect on the user's experience, and correctness matters far more — hence the heavy tilt toward capability in the weighting.

### The resulting shortlist

Applying this filter-then-rank process across all 146 models produced the following top 10 (note: this pass used a working budget of ₹5 lakh per month rather than the ₹3 lakh figure settled on earlier, which is why a few entries here run above that stricter number):

| Rank | Model | Rating | ₹/month | Speed (chars/sec) | Weighted score |
|---|---|---|---|---|---|
| 1 | GPT-5.6 Terra | 46.0 | ₹4,47,300 | 57 | 0.909 |
| 2 | Kimi K3 | 44.9 | ₹4,83,084 | 4 | 0.883 |
| 3 | ~~Muse Spark 1.1~~ | 38.3 | ₹1,65,501 | 266 | 0.823 |
| 4 | Grok 4.5 | 40.5 | ₹2,50,488 | 24 | 0.817 |
| 5 | Claude Sonnet 5 | 40.4 | ₹2,84,760 | 15 | 0.814 |
| 6 | GPT-5.6 Luna | 39.3 | ₹1,78,920 | 22 | 0.798 |
| 7 | GLM-5.2 | 39.0 | ₹1,21,666 | 9 | 0.791 |
| 8 | Qwen3.7 Max | 38.9 | ₹1,56,555 | 5 | 0.789 |
| 9 | MiniMax M3 | 35.4 | ₹42,941 | 171 | 0.761 |
| 10 | Gemini 3.6 Flash | 32.8 | ₹2,41,542 | 265 | 0.736 |

Rank 3, Muse Spark 1.1, gets struck out here for a practical reason unrelated to its score: it's only available for deployment in one specific region, ruling it out for an application that needs to serve users globally.

A genuinely interesting pattern shows up in this table: the paid, proprietary models (like GPT-5.6 Terra and Kimi K3) top the ranking, but several open-source models — MiniMax M3 in particular — land not far behind on score while costing a small fraction of the price. GPT-5.6 Terra combines strong capability and speed for 0.909, while MiniMax M3 reaches 0.761 at roughly a tenth of the monthly cost. Whether that gap in score justifies the difference in price is exactly the kind of question a custom eval — not a leaderboard — is built to answer.

### Choosing which candidates to actually test

Testing every viable candidate from the shortlist is the ideal approach, and nothing prevents it — but for a focused walkthrough, five candidates make a representative set:

- **GPT-5.6 Terra** — the top-ranked candidate overall, despite being the priciest.
- **Kimi K3** — a strong second-place candidate worth testing directly.
- **Grok 4.5** — representing that model family in the comparison.
- **Claude Sonnet 5** — the only Anthropic-family model that made the cost-filtered shortlist.
- **MiniMax M3** — chosen to represent the cluster of much cheaper open-source options, specifically because of how competitive its score is relative to its cost.

A few shortlisted models were deliberately left out of this particular test set: GPT-5.6 Luna, since it comes from the same family as Terra and is likely to underperform it, making both somewhat redundant to test; and Gemini 3.6 Flash, simply for the sake of keeping the comparison focused. This completes Stage 2 — five to ten candidates, narrowed down from the full leaderboard using this application's own requirements. Stage 3, running an actual custom evaluation on this shortlist, is next.

## Stage 3: running the custom evaluation

With five candidates shortlisted, the final and most decisive stage is testing them directly on real data and a real task. This stage needs two things beyond the candidates themselves: an actual database to query, and a **golden dataset** — a set of questions paired with their known-correct SQL queries.

### What a golden dataset looks like here

A golden dataset for this task is a set of natural-language questions — 30 to 50 is a reasonable range — each paired with a SQL query that a data analyst has written and personally verified against the real database. A question like "How many centuries did Virat Kohli score between 2018 and 2022?" gets its own hand-written, validated SQL query sitting right next to it.

Building this dataset well means keeping it genuinely representative of the real distribution of questions users actually ask — a mix of easy, medium, and hard difficulty rather than all one level, and deliberate variety in the kinds of SQL constructs involved: some questions requiring joins across tables, some requiring subqueries, some requiring window functions, and so on. A golden dataset skewed toward only one difficulty level or one query pattern won't give an honest picture of how a model performs on the actual range of questions it will face in production.

### Why comparing raw query text doesn't work

A natural first instinct for checking whether a model's generated SQL is "correct" is comparing it character by character against the golden query. This doesn't work, for a simple reason: the same result can be reached through genuinely different SQL queries. A model might write a query with a different join order, a different subquery structure, or a different way of filtering rows — textually nothing like the golden query — and still arrive at exactly the right answer.

The fix is to compare **results**, not **text**: run both the generated query and the golden query against the actual database, and compare the two resulting tables directly. If the two result sets match, the generated query is correct — regardless of how differently it was written. If they don't match, it's marked incorrect.

### The full evaluation loop

Put together, evaluating one candidate model against the golden dataset works like this: take each question in the golden dataset, send it to the model, and receive a generated SQL query back. Run that generated query against the database, and separately run the golden dataset's own verified query against the same database. Compare the two result tables — a match means the question was answered correctly, a mismatch means it wasn't. Repeat this for every question in the golden dataset, and the fraction answered correctly becomes that model's accuracy score.

Repeating this entire process once per candidate model produces a directly comparable accuracy figure for all five candidates on the exact same task and data — which is the number that actually decides which model gets used, rather than any benchmark or leaderboard score.

### How large should the golden dataset be?

There's no hard limit on golden dataset size, but two real costs grow alongside it. First, golden datasets are typically built by a human analyst, which costs time, effort, and money regardless of how large the set gets. Second, evaluation cost scales directly with size: 50 questions across 5 candidate models means 250 total model calls, but 500 questions across the same 5 models means 2,500 — a real, budgeted cost in its own right, not a free step. A range of roughly 50 to 500 questions is a reasonable planning target for most applications, and the same self-improving loop covered earlier applies here too: once a system is deployed, real production failures get folded back into the golden dataset over time, growing it naturally rather than trying to anticipate every case upfront.

For a focused walkthrough here, the golden dataset was deliberately kept small — just 20 questions, and deliberately skewed toward difficult ones specifically to make the gap between strong and weak models more visible. A production-grade golden dataset would include a proper mix of easy, medium, and hard questions rather than difficult ones exclusively.

## Building it: the practical steps

**Step 1 — the dataset.** The underlying data used here is real, publicly available ball-by-ball IPL data spanning 2008 through 2024, sourced from Kaggle. To keep the evaluation focused, the data was trimmed down to just 2020 through 2024.

**Step 2 — filter and load the database.** That trimmed data gets loaded from its raw CSV files into an actual SQLite database — two tables, matches and deliveries, exactly matching the schema described earlier — so real SQL queries can actually be run against it during evaluation.

### Step 3 — extract the schema

**Extracting the schema.** The database's structure — table names, column names, and column types — gets extracted directly from the live database into its own schema file. This is what gets inserted into the system prompt on every request, so the model always knows exactly what tables and columns are available to query against.

### Steps 4–5 — build and validate the golden dataset

**Step 4 — build the golden dataset.** Ideally, a golden dataset like this gets written entirely by a human data analyst. For this walkthrough, an LLM was used to generate a set of 20 deliberately difficult questions along with their corresponding SQL queries instead, standing in for the analyst's manual work.

**Step 5 — validate the golden dataset.** Every one of those 20 generated queries then gets run directly against the real database, specifically to catch anything that doesn't execute — a malformed query, a reference to a column that doesn't exist, and so on. All 20 ran without error. It's worth being precise about what this step actually confirms: it verifies the queries are *structurally valid* SQL that runs successfully, not that each one is *semantically* the exact right answer to its paired question — that trust comes from how the questions and queries were generated and reviewed, not from this validation step alone.

### Step 6 — store the golden dataset in CSV format

**Storing the golden dataset.** The 20 validated question-and-query pairs get extracted into a single CSV file — the actual golden dataset going forward. Alongside each question and its query, a couple of extra fields get stored too: the expected number of result rows, and whether the query's correctness depends on row order (whether an `ORDER BY` clause matters for this particular question) — both useful details for scoring results correctly later on.

### Step 7 — check the flow end to end

**Checking the flow.** Before running the real evaluation across all five candidates, it's worth confirming the entire pipeline actually works — pick one question from the golden dataset, send it to a model (a throwaway model, not one of the five actual candidates, used purely to sanity-check the setup), and confirm a valid SQL query comes back. This single test is what confirms the system prompt, the schema injection, and the model-calling code are all wired together correctly before spending time and money running the full evaluation.

## A practical tool: OpenRouter

The five candidate models come from different providers — OpenAI, Anthropic, xAI, and a couple of Chinese labs — each of which would normally need its own separate API integration code. **OpenRouter** solves this directly: it's a platform that provides unified API access to a very wide range of models across providers through one consistent interface, meaning the same integration code works regardless of which underlying model is being called. It also integrates directly with LangChain, the same way a native provider integration would.

OpenRouter offers free credits for testing, though running a full five-model evaluation like this one still costs a small amount directly — in this case, roughly $5 (around ₹500) in total.

### Step 8 — pick candidate models

**Picking candidate models.** With OpenRouter as the access layer, the five shortlisted candidates get compiled into a simple list, each paired with its exact model identifier ("slug") as OpenRouter recognizes it — the specific string needed to actually call that model through the API. This list is what the evaluation loop will iterate through once it's built.

### Step 9 — build the evaluator

**The comparison algorithm.** Turning "compare two result tables" into actual working logic takes a few distinct checks, run in order:

1. **Compare row counts first.** If the generated query's result has a different number of rows than the golden query's result, the comparison stops right there — two result sets with different row counts cannot represent the same correct answer, regardless of what values they contain.
2. **Normalize values before comparing.** Two representations of the same underlying value shouldn't be treated as a mismatch — `2.0` and `2` should count as identical, and so should two floating-point values that differ only by a tiny rounding amount, like `2.999` and `2.99`. Without this step, a technically correct query could get marked wrong purely due to superficial formatting differences.
3. **Handle row order correctly.** Sometimes a query can return the exact right values but in a different row order purely due to how it was written — this doesn't necessarily mean it's wrong. If the golden dataset indicates that order doesn't matter for a given question (no `ORDER BY` clause required), both result sets get sorted before comparing, so reordering doesn't cause a false mismatch. If order *does* matter for that question, the two result sets are compared directly, without sorting — matching in that case confirms the model correctly used the right ordering logic, and a mismatch means it didn't.

A few additional checks exist in the actual implementation beyond these three, but this covers the core logic: same row count, normalized values, and order handled according to what each specific question actually requires.

### Step 10 — build the orchestrator

The final piece of code ties every earlier step together into one runnable process: load the schema and the golden dataset, connect to the database, then loop through each of the five candidate models one at a time. For each model, loop through every question in the golden dataset — call the model, take its generated SQL, execute both the generated query and the golden query against the database, pass both results to the evaluator logic above, and log whether they matched. Once every question has been run for every model, the accumulated results produce a full accuracy score per candidate.

### Steps 11–12 — running the evaluation and reading the results

Running this orchestrator end to end works through all five candidates, one after another, streaming a running tally of matches and mismatches for each. The final accuracy figures that came out of this run:

| Model | Accuracy | Monthly cost | Notable observations |
|---|---|---|---|
| GPT-5.6 Terra | 80% | ₹4,47,300 | Solid accuracy, but the most expensive candidate by a wide margin |
| Kimi K3 | 55% | ₹4,83,084 | Notably slow, and several failures were outright SQL syntax errors rather than just wrong answers |
| **Grok 4.5** | **90%** | **₹2,50,488** | Fastest of the strong performers, no syntax errors, best accuracy overall |
| Claude Sonnet 5 | 85% | ₹2,84,760 | Very fast, no syntax errors, close second on both accuracy and cost |
| MiniMax M3 | 65% | ₹42,941 | Extremely cheap, but multiple SQL syntax errors brought accuracy down significantly |

Two failure types showed up during this run, and it's worth keeping them conceptually separate: a **mismatch**, where the generated query runs successfully but returns the wrong result, versus a **SQL syntax error**, where the generated query doesn't even execute — a more basic kind of failure. Syntax errors showed up specifically on the candidates that otherwise scored lowest here, which is a meaningfully worse failure mode than simply getting an answer wrong.

### A concrete lesson: leaderboard hype doesn't guarantee task performance

Kimi K3 had generated significant buzz prior to this evaluation — a large, openly-weighted model from a Chinese AI lab that had drawn considerable attention for its capability relative to its cost. Despite that reputation, it turned out to be both the slowest and the least accurate of the five candidates on this specific task, with several outright syntax errors along the way.

This is exactly the caution raised earlier about leaderboards and hype: a model performing impressively on general benchmarks, or generating significant buzz in the wider field, doesn't guarantee it will perform well on one specific, real task. The only way to know is to actually test it — which is the entire premise of running a custom eval in the first place, rather than trusting a leaderboard position or reputation alone.

## Making the decision

With the results in hand, the field narrows quickly. GPT-5.6 Terra is ruled out on cost — its accuracy doesn't justify being nearly twice the price of the next-best options. Kimi K3 and MiniMax M3 are ruled out on accuracy, both falling well short of the top performers despite MiniMax's very low cost.

That leaves two finalists: **Grok 4.5** and **Claude Sonnet 5** — close on both accuracy (90% vs. 85%) and cost (₹2.5 lakh vs. ₹2.85 lakh), with similar general capability ratings on the coding leaderboard used earlier as well. When two candidates land this close together on the numbers, the final call often comes down to considerations the eval itself doesn't capture directly — a team's past experience with a particular provider's API reliability, existing infrastructure or contracts, or simply a team vote when the quantitative case doesn't clearly favor one option over the other. Both of these outcomes are legitimate ways to resolve a genuine statistical tie; which one wins out is a judgment call for the team making the final decision, not something the eval result alone dictates.

A caveat worth stating plainly: this was intentionally a small, difficulty-skewed golden dataset of just 20 questions, where a single question's outcome shifts the accuracy figure by a full 5 percentage points. A production-grade evaluation would use a larger, more representative golden dataset, and ideally run multiple independent passes over it — since each model call is independent, repeating the full evaluation several times and averaging the results provides meaningfully stronger statistical confidence that a given accuracy figure isn't just a product of which specific 20 questions happened to be asked.

## Closing this case study

This walkthrough completes the full three-stage model-selection process end to end: gathering requirements, shortlisting candidates from a leaderboard, and running a genuine custom evaluation to make the final call — exactly the workflow that should replace "just pick whatever's ranked highest" for any real application.

This also closes out the broader model-evaluations arc — benchmarks, benchmark anatomy, eval harnesses, leaderboards, and now custom model evals applied end to end on a real task. The natural next step from here is **application evals**: evaluating not just the underlying model, but the full RAG and agentic systems built on top of it.